In [1]:
import json
from datetime import datetime

INPUT_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\conversations.json"
OUTPUT_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\conversations_truncated.json"
TARGET_N = 300_000  # ajustable

def truncate_conversations_by_recency(input_path, output_path, target_n):
    with open(input_path, "r", encoding="utf-8") as f:
        convs = json.load(f)  # dict {conv_id: {title, timestamp, ...}}

    print(f"Total de posts original: {len(convs)}")

    # Ordenamos por timestamp descendente (más reciente primero)
    sorted_items = sorted(
        convs.items(),
        key=lambda kv: kv[1].get("timestamp", 0),
        reverse=True
    )

    truncated_items = sorted_items[:target_n]
    truncated_dict = dict(truncated_items)

    # Sanity check: rango temporal cubierto por la muestra
    timestamps = [meta.get("timestamp", 0) for _, meta in truncated_items]
    oldest = datetime.utcfromtimestamp(min(timestamps))
    newest = datetime.utcfromtimestamp(max(timestamps))
    print(f"Posts conservados: {len(truncated_dict)}")
    print(f"Rango temporal: {oldest} → {newest}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(truncated_dict, f, ensure_ascii=False)

    print(f"Guardado en: {output_path}")
    return truncated_dict

sampled_conv_ids = set(
    truncate_conversations_by_recency(INPUT_PATH, OUTPUT_PATH, TARGET_N).keys()
)

Total de posts original: 627372
Posts conservados: 300000
Rango temporal: 2017-10-27 02:29:38 → 2018-10-31 23:59:38
Guardado en: D:\TFM\data\NoStupidQuestions.corpus\conversations_truncated.json


In [2]:
CONV_TRUNCATED_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\conversations_truncated.json"
UTTERANCES_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\utterances.jsonl"
OUTPUT_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\utterances_truncated.jsonl"

def load_sampled_ids(conv_truncated_path):
    with open(conv_truncated_path, "r", encoding="utf-8") as f:
        convs = json.load(f)
    return set(convs.keys())

def truncate_utterances(utterances_path, output_path, sampled_ids):
    n_seen = 0
    n_kept = 0
    n_posts_kept = 0

    with open(utterances_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            n_seen += 1
            utt = json.loads(line)

            if utt.get("root") in sampled_ids:
                fout.write(line)  # escribimos la línea cruda tal cual, sin reserializar
                n_kept += 1
                if utt.get("reply_to") is None:
                    n_posts_kept += 1

            if n_seen % 1_000_000 == 0:
                print(f"Procesadas {n_seen:,} líneas... conservadas: {n_kept:,}")

    print(f"\nTotal líneas procesadas: {n_seen:,}")
    print(f"Total utterances conservadas (posts+comentarios): {n_kept:,}")
    print(f"De ellas, posts: {n_posts_kept:,}")
    print(f"Guardado en: {output_path}")

sampled_ids = load_sampled_ids(CONV_TRUNCATED_PATH)
print(f"Ids de conversación cargados: {len(sampled_ids):,}")

truncate_utterances(UTTERANCES_PATH, OUTPUT_PATH, sampled_ids)

Ids de conversación cargados: 300,000
Procesadas 1,000,000 líneas... conservadas: 300,000
Procesadas 2,000,000 líneas... conservadas: 300,000
Procesadas 3,000,000 líneas... conservadas: 587,226
Procesadas 4,000,000 líneas... conservadas: 1,586,972

Total líneas procesadas: 4,721,701
Total utterances conservadas (posts+comentarios): 2,308,673
De ellas, posts: 300,000
Guardado en: D:\TFM\data\NoStupidQuestions.corpus\utterances_truncated.jsonl


In [3]:
CONV_TRUNCATED_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\conversations_truncated.json"
UTT_TRUNCATED_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\utterances_truncated.jsonl"

# 1. Ids esperados desde conversations_truncated.json
with open(CONV_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    conv_ids = set(json.load(f).keys())

print(f"Ids en conversations_truncated.json: {len(conv_ids):,}")

# 2. Recorremos utterances_truncated.jsonl y separamos posts vs comentarios
post_ids = []
all_roots = set()
n_total = 0

with open(UTT_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        utt = json.loads(line)
        n_total += 1
        all_roots.add(utt["root"])
        if utt.get("reply_to") is None:
            post_ids.append(utt["id"])

post_ids_set = set(post_ids)

print(f"Total utterances en el fichero truncado: {n_total:,}")
print(f"Posts encontrados (reply_to is None): {len(post_ids):,}")
print(f"Posts únicos (set): {len(post_ids_set):,}")

# 3. Comprobaciones clave
duplicados = len(post_ids) - len(post_ids_set)
faltantes_en_utt = conv_ids - post_ids_set      # ids en conversations pero sin post en utterances
sobrantes_en_utt = post_ids_set - conv_ids      # posts en utterances que no estaban en conversations
roots_fuera_de_conv = all_roots - conv_ids      # cualquier utterance (post o comentario) con root inesperado

print(f"\n--- Resultados de la comprobación ---")
print(f"Duplicados en post_ids: {duplicados}")
print(f"Ids en conversations SIN post correspondiente en utterances: {len(faltantes_en_utt)}")
print(f"Posts en utterances que NO estaban en conversations: {len(sobrantes_en_utt)}")
print(f"Roots (posts+comentarios) fuera del set esperado: {len(roots_fuera_de_conv)}")

coincide_exacto = (conv_ids == post_ids_set)
print(f"\n¿Coinciden exactamente los sets de ids?: {coincide_exacto}")

Ids en conversations_truncated.json: 300,000
Total utterances en el fichero truncado: 2,308,673
Posts encontrados (reply_to is None): 300,000
Posts únicos (set): 300,000

--- Resultados de la comprobación ---
Duplicados en post_ids: 0
Ids en conversations SIN post correspondiente en utterances: 0
Posts en utterances que NO estaban en conversations: 0
Roots (posts+comentarios) fuera del set esperado: 0

¿Coinciden exactamente los sets de ids?: True


In [4]:
import pandas as pd

CONV_TRUNCATED_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\conversations_truncated.json"
UTT_TRUNCATED_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\utterances_truncated.jsonl"
OUTPUT_PATH = "D:\\TFM\\data\\NoStupidQuestions.corpus\\NoStupidQuestions_posts_final.parquet"

def load_conversations(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)  # dict {id: {title, num_comments, domain, timestamp, ...}}

def extract_posts_only(utt_path):
    """Filtra utterances_truncated.jsonl quedándonos solo con los posts (reply_to is None)."""
    posts = []
    n_seen = 0
    with open(utt_path, "r", encoding="utf-8") as f:
        for line in f:
            n_seen += 1
            utt = json.loads(line)
            if utt.get("reply_to") is not None:
                continue  # es comentario, lo descartamos
            posts.append({
                "id": utt["id"],
                "user": utt.get("user"),
                "body_text": utt.get("text", ""),
                "timestamp": utt.get("timestamp"),
                "score": utt.get("meta", {}).get("score"),
            })
    print(f"Líneas totales revisadas: {n_seen:,}")
    print(f"Posts extraídos: {len(posts):,}")
    return posts

# 1. Carga metadata de conversaciones (título, domain, num_comments...)
conv_meta = load_conversations(CONV_TRUNCATED_PATH)

# 2. Extrae solo los posts del jsonl truncado
post_rows = extract_posts_only(UTT_TRUNCATED_PATH)
df_posts = pd.DataFrame(post_rows)

# 3. Construye dataframe de metadata de conversaciones
df_meta = pd.DataFrame([
    {"id": cid, **meta} for cid, meta in conv_meta.items()
])

# 4. Merge por id
df_final = df_posts.merge(df_meta, on="id", how="left", validate="one_to_one")

# 5. Texto final: título + cuerpo
df_final["full_text"] = (
    df_final["title"].fillna("") + " " + df_final["body_text"].fillna("")
).str.strip()

# 6. Sanity checks antes de guardar
print(f"\nFilas finales: {len(df_final):,}")
print(f"Nulos en title: {df_final['title'].isna().sum()}")
print(f"Nulos en body_text: {(df_final['body_text'] == '').sum()} (texto vacío)")
print(f"Filas con full_text vacío tras strip: {(df_final['full_text'] == '').sum()}")

df_final.to_parquet(OUTPUT_PATH, index=False)
print(f"\nGuardado en: {OUTPUT_PATH}")

Líneas totales revisadas: 2,308,673
Posts extraídos: 300,000

Filas finales: 300,000
Nulos en title: 0
Nulos en body_text: 74913 (texto vacío)
Filas con full_text vacío tras strip: 0

Guardado en: D:\TFM\data\NoStupidQuestions.corpus\NoStupidQuestions_posts_final.parquet


In [8]:
import pandas as pd
from datetime import datetime

df = pd.read_parquet("D:\\TFM\\data\\NoStupidQuestions.corpus\\NoStupidQuestions_posts_final.parquet")

# 1. Estructura básica
print(df.shape)
print(df.dtypes)
df.info()

(300000, 15)
id                      str
user                    str
body_text               str
timestamp_x           int64
score                 int64
title                   str
num_comments          int64
domain                  str
timestamp_y           int64
subreddit               str
gilded                int64
gildings             object
stickied               bool
author_flair_text       str
full_text               str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   id                 300000 non-null  str   
 1   user               300000 non-null  str   
 2   body_text          300000 non-null  str   
 3   timestamp_x        300000 non-null  int64 
 4   score              300000 non-null  int64 
 5   title              300000 non-null  str   
 6   num_comments       300000 non-null  int64 
 7   domain          

In [9]:
df.head(3)

,id,user,body_text,timestamp_x,score,title,num_comments,domain,timestamp_y,subreddit,gilded,gildings,stickied,author_flair_text,full_text
0,8nncko,[deleted],[deleted],1527811206,0,Do/ can blind people get dizzy? How?,5,self.NoStupidQuestions,1527811206,NoStupidQuestions,0,None,False,,Do/ can blind people get dizzy? How? [deleted]
1,8nncsl,energy_sync,"Why doesn't the rain just come out gradually, ...",1527811257,1,Why does rain come out of clouds all at once?,4,self.NoStupidQuestions,1527811257,NoStupidQuestions,0,None,False,,Why does rain come out of clouds all at once? ...
2,8nnd07,Ronanfob,Is there any number of years in which a basket...,1527811310,2,If I were to bounce a basketball in the exact ...,2,self.NoStupidQuestions,1527811310,NoStupidQuestions,0,None,False,,If I were to bounce a basketball in the exact ...
